## What this notebook teaches

This notebook is mainly about **scikit-learn**. Bybit market data is used.

By the end, you should understand the practical sklearn workflow:

- build a table of features `X` and a target `y`
- split data without leaking the future into the past
- combine preprocessing and models with `Pipeline`
- call `fit`, `predict`, and `predict_proba`
- evaluate classification and regression models with metrics
- inspect data and model behavior

> Not financial advice and not a profitable trading strategy.

## Tiny finance vocabulary for this notebook

We will keep finance terms short and practical.

> **USDT.** USDT is a dollar-pegged crypto token commonly used as the quote currency on crypto exchanges.

> **perpetual contract.** A perpetual contract is a crypto derivative with no expiration date. In this notebook we only use its public price candles as learning data.

> **candle or candlestick.** A candle summarizes price movement during one time interval, such as one hour.

> **OHLCV.** OHLCV means open, high, low, close, and volume. Open is the first price in the interval, high is the maximum, low is the minimum, close is the last price, and volume is the traded amount.

> **turnover.** Turnover is the traded value during the interval. For USDT contracts, Bybit reports it in quote currency such as USDT.

In [ ]:
from __future__ import annotations

import time
from typing import Any

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from IPython.display import HTML
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.options.display.float_format = "{:.3f}".format
px.defaults.template = "plotly_white"

## 1. Download public Bybit candles

The Bybit V5 endpoint `/v5/market/kline` returns historical candles. We will use `category="linear"`, `interval="60"`, and the symbols `BTCUSDT`, `ETHUSDT`, and `SOLUSDT`.

The API returns at most 1000 candles per request, so the downloader below paginates backward from the end date.

In [ ]:
BYBIT_KLINE_URL = "https://api.bybit.com/v5/market/kline"
CATEGORY = "linear"
INTERVAL = "60"
INTERVAL_MS = 60 * 60 * 1000
LIMIT = 1000
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]
TARGET_SYMBOL = "BTCUSDT"
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"
RANDOM_STATE = 42

KLINE_COLUMNS = [
    "start_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "turnover",
]


def utc_ms(date_text: str) -> int:
    """Convert a YYYY-MM-DD date to a UTC millisecond timestamp."""
    return int(pd.Timestamp(date_text, tz="UTC").timestamp() * 1000)


def show_plot(fig: go.Figure, height: int = 500) -> HTML:
    """Render Plotly as saved HTML so Quarto can publish the output."""
    fig.update_layout(
        height=height,
        margin={"l": 40, "r": 30, "t": 70, "b": 40},
    )
    return HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


def request_kline_page(
    session: requests.Session,
    symbol: str,
    start_ms: int,
    end_ms: int,
) -> list[list[str]]:
    """Request one page of Bybit candles with light retry logic."""
    params: dict[str, Any] = {
        "category": CATEGORY,
        "symbol": symbol,
        "interval": INTERVAL,
        "start": start_ms,
        "end": end_ms,
        "limit": LIMIT,
    }

    for attempt in range(5):
        response = session.get(BYBIT_KLINE_URL, params=params, timeout=30)
        if response.status_code == 429:
            time.sleep(2**attempt)
            continue

        response.raise_for_status()
        payload = response.json()
        if payload.get("retCode") == 0:
            return payload["result"]["list"]

        if attempt == 4:
            raise RuntimeError(f"Bybit returned an error: {payload}")
        time.sleep(2**attempt)

    return []


def rows_to_frame(rows: list[list[str]], symbol: str) -> pd.DataFrame:
    """Convert raw Bybit kline rows into a typed pandas DataFrame."""
    frame = pd.DataFrame(rows, columns=KLINE_COLUMNS)
    for column in KLINE_COLUMNS:
        frame[column] = pd.to_numeric(frame[column])

    frame["timestamp"] = pd.to_datetime(frame["start_time"], unit="ms", utc=True)
    frame["symbol"] = symbol
    return frame[
        ["timestamp", "symbol", "open", "high", "low", "close", "volume", "turnover"]
    ]


def fetch_bybit_klines(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetch all hourly candles for one symbol in the requested date range."""
    start_ms = utc_ms(start_date)
    end_ms = utc_ms(end_date)
    cursor_end = end_ms
    all_rows: list[list[str]] = []

    with requests.Session() as session:
        while cursor_end >= start_ms:
            rows = request_kline_page(session, symbol, start_ms, cursor_end)
            if not rows:
                break

            all_rows.extend(rows)
            oldest_start = min(int(row[0]) for row in rows)
            if oldest_start <= start_ms:
                break

            cursor_end = oldest_start - INTERVAL_MS
            time.sleep(0.05)

    frame = rows_to_frame(all_rows, symbol)
    return (
        frame.drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .query("@start_timestamp <= timestamp <= @end_timestamp")
        .reset_index(drop=True)
    )

In [21]:
frames = []
for symbol in SYMBOLS:
    frame = fetch_bybit_klines(symbol, START_DATE, END_DATE)
    print(
        f"{symbol}: {len(frame):,} candles from "
        f"{frame['timestamp'].min()} to {frame['timestamp'].max()}"
    )
    frames.append(frame)

candles = pd.concat(frames, ignore_index=True).sort_values(["symbol", "timestamp"])
candles.head()

BTCUSDT: 17,521 candles from 2024-01-01 00:00:00+00:00 to 2025-12-31 00:00:00+00:00
ETHUSDT: 17,521 candles from 2024-01-01 00:00:00+00:00 to 2025-12-31 00:00:00+00:00
SOLUSDT: 17,521 candles from 2024-01-01 00:00:00+00:00 to 2025-12-31 00:00:00+00:00


,timestamp,symbol,open,high,low,close,volume,turnover
0,2024-01-01 00:00:00+00:00,BTCUSDT,42324.800,42610.900,42300.200,42517.400,3038.207,129022621.257
1,2024-01-01 01:00:00+00:00,BTCUSDT,42517.400,42842.900,42475.100,42661.300,3308.231,141296220.369
2,2024-01-01 02:00:00+00:00,BTCUSDT,42661.300,42691.900,42550.800,42631.800,1651.447,70384296.447
3,2024-01-01 03:00:00+00:00,BTCUSDT,42631.800,42641.400,42271.500,42384.100,3499.277,148488084.347
4,2024-01-01 04:00:00+00:00,BTCUSDT,42384.100,42448.000,42250.500,42446.300,1800.157,76253111.486


## 2. Look at the raw market series

This first chart is only orientation.

> **close price.** The close is the last traded price inside a candle interval. We use hourly closes to compute returns.

In [22]:
btc_candles = candles.query("symbol == @TARGET_SYMBOL").copy()

fig = px.line(
    btc_candles,
    x="timestamp",
    y="close",
    title="BTCUSDT hourly close price from Bybit",
    labels={"timestamp": "Time", "close": "Close price (USDT)"},
)
fig.update_xaxes(rangeslider_visible=True)
show_plot(fig)

## 3. Build `X` and `y`

In sklearn, most supervised learning starts with two objects: `X`, the feature table, and `y`, the target we want the model to learn. Here one row means "information available after an hourly candle closes." The target is the **next** hourly BTC return.

> **return.** A return measures relative price change. If BTC moves from 100 to 101, the return is `(101 / 100) - 1 = 0.01`, or 1%.

> **volatility.** Volatility is how much returns move around. A simple rolling standard deviation is a common beginner-friendly estimate.

> **volume.** Volume is how much of the asset traded during the candle.

In [24]:
def pivot_candles(candles_frame: pd.DataFrame, value: str) -> pd.DataFrame:
    """Pivot long candle data to timestamp x symbol format."""
    return candles_frame.pivot(
        index="timestamp", columns="symbol", values=value
    ).sort_index()


def pct_change(series_or_frame: pd.Series | pd.DataFrame, periods: int = 1):
    """Pandas percent change without forward-filling missing values."""
    return series_or_frame.pct_change(periods=periods, fill_method=None)


def build_learning_table(candles_frame: pd.DataFrame) -> pd.DataFrame:
    """Create sklearn-ready features and next-hour targets."""
    close = pivot_candles(candles_frame, "close")
    high = pivot_candles(candles_frame, "high")
    low = pivot_candles(candles_frame, "low")
    volume = pivot_candles(candles_frame, "volume")
    turnover = pivot_candles(candles_frame, "turnover")

    returns = pct_change(close)
    btc_returns = returns[TARGET_SYMBOL]

    features = pd.DataFrame(index=close.index)
    features["btc_return_1h"] = btc_returns
    features["btc_return_3h"] = pct_change(close[TARGET_SYMBOL], periods=3)
    features["btc_return_12h"] = pct_change(close[TARGET_SYMBOL], periods=12)
    features["btc_rolling_mean_24h"] = btc_returns.rolling(24).mean()
    features["btc_volatility_24h"] = btc_returns.rolling(24).std()
    features["btc_candle_range_pct"] = (
        high[TARGET_SYMBOL] - low[TARGET_SYMBOL]
    ) / close[TARGET_SYMBOL]
    features["btc_volume_change_1h"] = pct_change(volume[TARGET_SYMBOL])
    features["btc_turnover_change_1h"] = pct_change(turnover[TARGET_SYMBOL])

    for symbol in ["ETHUSDT", "SOLUSDT"]:
        prefix = symbol.replace("USDT", "").lower()
        features[f"{prefix}_return_1h"] = returns[symbol]
        features[f"{prefix}_return_12h"] = pct_change(close[symbol], periods=12)

    features["btc_minus_eth_return_1h"] = (
        features["btc_return_1h"] - features["eth_return_1h"]
    )
    features["btc_minus_sol_return_1h"] = (
        features["btc_return_1h"] - features["sol_return_1h"]
    )

    learning_table = features.copy()
    learning_table["target_return_next_hour"] = btc_returns.shift(-1)
    learning_table["target_up_next_hour"] = (
        learning_table["target_return_next_hour"] > 0
    ).astype(int)

    return learning_table.replace([np.inf, -np.inf], np.nan).dropna()


learning_data = build_learning_table(candles)
feature_columns = [
    column for column in learning_data.columns if not column.startswith("target_")
]
target_columns = ["target_return_next_hour", "target_up_next_hour"]

print(f"Rows: {len(learning_data):,}")
print(f"Features: {len(feature_columns)}")
learning_data[feature_columns + target_columns].head()

Rows: 17,496
Features: 14


,btc_return_1h,btc_return_3h,btc_return_12h,btc_rolling_mean_24h,btc_volatility_24h,btc_candle_range_pct,btc_volume_change_1h,btc_turnover_change_1h,eth_return_1h,eth_return_12h,sol_return_1h,sol_return_12h,btc_minus_eth_return_1h,btc_minus_sol_return_1h,target_return_next_hour,target_up_next_hour
timestamp,,,,,,,,,,,,,,,,
2024-01-02 00:00:00+00:00,0.022,0.035,0.058,0.003,0.006,0.026,0.911,0.943,0.016,0.039,0.006,0.056,0.005,0.015,-0.006,0
2024-01-02 01:00:00+00:00,-0.006,0.031,0.051,0.002,0.006,0.017,-0.523,-0.519,-0.004,0.032,0.005,0.056,-0.001,-0.010,0.011,1
2024-01-02 02:00:00+00:00,0.011,0.027,0.065,0.003,0.006,0.011,-0.434,-0.433,0.001,0.035,0.007,0.072,0.010,0.004,0.000,1
2024-01-02 03:00:00+00:00,0.000,0.005,0.061,0.003,0.006,0.010,0.453,0.459,0.001,0.031,-0.005,0.052,-0.001,0.005,-0.005,0
2024-01-02 04:00:00+00:00,-0.005,0.006,0.057,0.003,0.006,0.006,-0.481,-0.482,0.000,0.033,0.009,0.055,-0.005,-0.014,0.000,1


## 4. Inspect features before modeling

A good sklearn workflow usually includes quick checks before model training. We will inspect the target distribution and feature correlations. This is still sklearn thinking: before fitting an estimator, understand what your table contains.

In [6]:
returns_plot_data = learning_data[["target_return_next_hour"]].copy()
returns_plot_data["target_return_percent"] = (
    returns_plot_data["target_return_next_hour"] * 100
)

fig = px.histogram(
    returns_plot_data,
    x="target_return_percent",
    nbins=120,
    title="Distribution of next-hour BTC returns",
    labels={
        "target_return_percent": "Next-hour return (%)",
        "count": "Number of hours",
    },
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
show_plot(fig)

In [25]:
correlation_data = learning_data[feature_columns + ["target_return_next_hour"]].corr()

fig = px.imshow(
    correlation_data,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    text_auto=".2f",
    title="Feature correlation matrix",
    labels={"color": "Correlation"},
)
fig.update_xaxes(tickangle=45)
show_plot(fig, height=760)

## 5. Train/test split without time leakage

Random train/test splits are common in beginner examples, but time series need a chronological split. If we randomly shuffle rows, the model may learn from the future and look better than it is.

In sklearn terms, `fit` must see only the training period, `predict` evaluates later unseen rows, and transformers such as `StandardScaler` must also be fitted only on training data. `Pipeline` helps us keep that discipline.

In [27]:
X = learning_data[feature_columns]
y_class = learning_data["target_up_next_hour"]
y_reg = learning_data["target_return_next_hour"]

split_index = int(len(learning_data) * 0.8)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train_class = y_class.iloc[:split_index]
y_test_class = y_class.iloc[split_index:]
y_train_reg = y_reg.iloc[:split_index]
y_test_reg = y_reg.iloc[split_index:]

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(X_train),
            "start": X_train.index.min(),
            "end": X_train.index.max(),
            "up_rate": y_train_class.mean(),
        },
        {
            "split": "test",
            "rows": len(X_test),
            "start": X_test.index.min(),
            "end": X_test.index.max(),
            "up_rate": y_test_class.mean(),
        },
    ]
)
split_summary

,split,rows,start,end,up_rate
0,train,13996,2024-01-02 00:00:00+00:00,2025-08-07 03:00:00+00:00,0.508
1,test,3500,2025-08-07 04:00:00+00:00,2025-12-30 23:00:00+00:00,0.502


## 6. Classification with an sklearn pipeline

First task: predict whether the next BTC return is positive.

`LogisticRegression` is a useful first classifier because it is fast and easy to inspect. `StandardScaler` rescales features so each feature has a comparable numerical scale.

~~~python
model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)
model.predict(X_test)
~~~

The pipeline behaves like one estimator, but internally it applies preprocessing before the model.

In [28]:
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
)

time_split = TimeSeriesSplit(n_splits=5)
cv_scores = cross_validate(
    logistic_model,
    X_train,
    y_train_class,
    cv=time_split,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "roc_auc": "roc_auc",
    },
)

cv_summary = (
    pd.DataFrame(cv_scores)
    .filter(like="test_")
    .agg(["mean", "std"])
    .T.rename(index=lambda value: value.replace("test_", ""))
)
cv_summary

,mean,std
accuracy,0.532,0.011
balanced_accuracy,0.532,0.011
roc_auc,0.543,0.012


## 7. Compare with a tree-based model

Now we compare the pipeline to a tree-based sklearn model. Tree models split the feature space into regions and usually do not need feature scaling.

This is not about finding the best trading model. It is about learning the sklearn habit: train more than one reasonable estimator and compare them on the same holdout period.

In [29]:
forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=25,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

models = {
    "Logistic regression pipeline": logistic_model,
    "Random forest classifier": forest_model,
}

classification_rows = []
test_predictions = pd.DataFrame(index=X_test.index)
test_predictions["actual_up"] = y_test_class

for name, model in models.items():
    model.fit(X_train, y_train_class)
    predicted = model.predict(X_test)
    probability_up = model.predict_proba(X_test)[:, 1]

    classification_rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test_class, predicted),
            "balanced_accuracy": balanced_accuracy_score(y_test_class, predicted),
            "roc_auc": roc_auc_score(y_test_class, probability_up),
        }
    )

    safe_name = name.lower().replace(" ", "_")
    test_predictions[f"{safe_name}_predicted_up"] = predicted
    test_predictions[f"{safe_name}_probability_up"] = probability_up

classification_metrics = pd.DataFrame(classification_rows).sort_values(
    "roc_auc", ascending=False
)
classification_metrics

,model,accuracy,balanced_accuracy,roc_auc
1,Random forest classifier,0.526,0.525,0.533
0,Logistic regression pipeline,0.515,0.516,0.523


## 9. Regression with Ridge

Classification asked: "Will the next return be positive?" Regression asks: "How large will the next return be?"

The sklearn workflow is almost the same. We change the estimator and the metrics: `Ridge` is linear regression with regularization, MAE measures average absolute error, RMSE penalizes larger mistakes more strongly, and R2 compares the model to a mean-prediction baseline.

In [30]:
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
ridge_model.fit(X_train, y_train_reg)

ridge_predictions = ridge_model.predict(X_test)
baseline_predictions = np.full(
    shape=len(y_test_reg),
    fill_value=y_train_reg.mean(),
    dtype=float,
)


def regression_metric_row(name: str, predicted: np.ndarray) -> dict[str, float | str]:
    """Collect regression metrics in one row."""
    return {
        "model": name,
        "mae": mean_absolute_error(y_test_reg, predicted),
        "rmse": mean_squared_error(y_test_reg, predicted) ** 0.5,
        "r2": r2_score(y_test_reg, predicted),
    }


regression_metrics = pd.DataFrame(
    [
        regression_metric_row("Ridge pipeline", ridge_predictions),
        regression_metric_row("Train-mean baseline", baseline_predictions),
    ]
)
regression_metrics

,model,mae,rmse,r2
0,Ridge pipeline,0.003,0.004,-0.004
1,Train-mean baseline,0.003,0.004,-0.001


In [31]:
regression_plot = pd.DataFrame(
    {
        "actual_next_return": y_test_reg,
        "predicted_next_return": ridge_predictions,
    },
    index=X_test.index,
).iloc[-500:]

fig = px.line(
    regression_plot.reset_index(names="timestamp"),
    x="timestamp",
    y=["actual_next_return", "predicted_next_return"],
    title="Ridge regression: actual vs predicted next-hour returns",
    labels={
        "timestamp": "Time",
        "value": "Return",
        "variable": "Series",
    },
)
show_plot(fig, height=560)

## Sklearn recap

The important lessons are about the machine learning workflow:

- `X` is the feature table and `y` is the target.
- Transformers such as `StandardScaler` learn preprocessing rules from training data.
- Estimators such as `LogisticRegression`, `RandomForestClassifier`, and `Ridge` learn patterns from `X` to predict `y`.
- `Pipeline` keeps preprocessing and modeling together so validation is cleaner.
- Time-series data should be split chronologically to reduce future leakage.
- Metrics describe model behavior, but they do not prove that a trading system is profitable.

The finance data gave us a realistic table. The sklearn workflow is the reusable skill.